# AI Gateway Request/Response Logging 測試

驗證 APIM AI Gateway 能否完整記錄每個請求的 input 與 output，涵蓋：
1. Chat Completions API (非 streaming + streaming)
2. Responses API (非 streaming + streaming)
3. Agent Service API (建立 assistant、thread、run)

## 前置條件
- 已執行 `scripts/deploy-logging.ps1` 佈署 Log Analytics + App Insights
- 已執行 `scripts/apply-policy.ps1` 套用 logging policy
- 擁有 APIM Subscription Key

## 1️⃣ 環境設定

In [ ]:
# !pip install openai azure-identity azure-ai-projects

import os, json, time, uuid
from getpass import getpass
from openai import OpenAI

# --- 測試環境資訊 ---
APIM_ENDPOINT   = "https://testaigw01.azure-api.net/kunlenewfoundry01/openai/v1/"
DEPLOYMENT_NAME = "Kimi-K2.5"
API_VERSION     = "2025-01-01-preview"

# 讀取 APIM Subscription Key（建議從環境變數讀取）
APIM_KEY = os.environ.get("APIM_SUBSCRIPTION_KEY") or getpass("APIM Subscription Key: ")

# 每次執行產生新的 run id，方便在 Log 中追蹤
RUN_ID = f"test-{uuid.uuid4().hex[:8]}"
print(f"RUN_ID = {RUN_ID}")

In [ ]:
# OpenAI client 指向 APIM Gateway
client = OpenAI(
    base_url=APIM_ENDPOINT,
    api_key=APIM_KEY,
    default_headers={
        "api-key": APIM_KEY,
        "x-run-id": RUN_ID,
    },
)

## 2️⃣ 測試 Chat Completions API

In [ ]:
# TC-01: Non-streaming chat completion
# 注意：Kimi-K2.5 是 reasoning model，需要較大的 max_tokens（reasoning 階段會吃很多 token）
response = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[
        {"role": "system", "content": "你是個說繁體中文的助手."},
        {"role": "user",   "content": "100 * 100 = ?"},
    ],
    max_tokens=4000,
)
msg = response.choices[0].message
# reasoning model 會把思考過程放在 reasoning_content，最終答案放在 content
reasoning = getattr(msg, "reasoning_content", None) or msg.model_dump().get("reasoning_content")
print("Content   :", msg.content)
print("Reasoning :", reasoning)
print("Finish    :", response.choices[0].finish_reason)
print("Tokens    :", response.usage.model_dump())

In [ ]:
# TC-02: Streaming chat completion
stream = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[
        {"role": "user", "content": "Count 1 to 5 slowly."},
    ],
    max_tokens=4000,
    stream=True,
    stream_options={"include_usage": True},
)

full_content = ""
full_reasoning = ""
for chunk in stream:
    if chunk.choices:
        delta = chunk.choices[0].delta
        # 一般內容
        if getattr(delta, "content", None):
            full_content += delta.content
            print(delta.content, end="", flush=True)
        # reasoning model 的思考過程
        reasoning_piece = getattr(delta, "reasoning_content", None) or delta.model_dump().get("reasoning_content")
        if reasoning_piece:
            full_reasoning += reasoning_piece
            print(f"\033[90m{reasoning_piece}\033[0m", end="", flush=True)  # 灰色顯示 reasoning
    if chunk.usage:
        print("\n\nUsage:", chunk.usage.model_dump())
print()
print(f"\n--- Summary ---\nContent length: {len(full_content)}  Reasoning length: {len(full_reasoning)}")

## 3️⃣ 測試 Responses API

In [ ]:
# TC-03: Non-streaming response
# Responses API 支援是否需視後端模型而定；若後端是 Azure OpenAI 模型且支援 /responses 才可用
try:
    resp = client.responses.create(
        model=DEPLOYMENT_NAME,
        input="Write a haiku about APIs.",
    )
    print("Response:", resp.output_text)
except Exception as e:
    print(f"Responses API not supported by deployment: {e}")

In [ ]:
# TC-04: Streaming response
try:
    stream = client.responses.create(
        model=DEPLOYMENT_NAME,
        input="Count from 1 to 3.",
        stream=True,
    )
    for event in stream:
        print(event.type, getattr(event, 'delta', ''), flush=True)
except Exception as e:
    print(f"Streaming Responses API not supported: {e}")

## 4️⃣ 測試 Agent Service API

> **注意**：Agent Service 通常透過 `azure-ai-projects` SDK 存取，需將 SDK 的 endpoint 指向 APIM。
> 這邊示範用 raw REST 呼叫展示 APIM 能否攔截並記錄 request body。

In [ ]:
import httpx

# TC-05: 建立 Assistant（實際 Agent Service 路徑可能不同，依你的 APIM 設定調整）
# 常見端點：POST /assistants
#          POST /threads
#          POST /threads/{id}/messages
#          POST /threads/{id}/runs

agent_base = APIM_ENDPOINT.rstrip('/')

# 示範請求（實際端點與 payload 以 Foundry Agent Service 文件為準）
print("⚠️  Agent Service 的完整端點路徑需依 APIM 中 agent API 的設定而定。")
print("    若 APIM 尚未建立 agent API，請先在 APIM 中 import Foundry Agent Service OpenAPI。")

## 5️⃣ 驗證日誌

測試完成後，等待 30–60 秒讓日誌匯入 Application Insights / Log Analytics，然後到 Azure Portal 執行：

**Log Analytics → Logs**
```kusto
ApiManagementGatewayLlmLog
| where TimeGenerated > ago(10m)
| project TimeGenerated, DeploymentName = tostring(Properties.deploymentName),
          PromptTokens = toint(Properties.promptTokens),
          CompletionTokens = toint(Properties.completionTokens),
          Messages = tostring(Properties.messages),
          Response = tostring(Properties.response)
| order by TimeGenerated desc
```

**Application Insights → Logs（查 custom traces）**
```kusto
traces
| where timestamp > ago(10m)
| where customDimensions.source in ("llm-logging", "agent-logging")
| project timestamp, message, customDimensions
```

更多查詢見 [`../kql/queries.kql`](../kql/queries.kql)

## ✅ 驗收清單

- [ ] Chat Completions (non-streaming) 的 prompt + completion 在 Log Analytics 可查
- [ ] Chat Completions (streaming) 的完整 output 已被組合記錄
- [ ] Responses API 的 input + output 已記錄
- [ ] Agent Service 的 request body 已透過 custom trace 記錄
- [ ] Token usage 正確
- [ ] 可用 `x-run-id` 關聯同一次測試的所有請求
- [ ] (新) Streaming 完整 content + reasoning 透過 client-side logging 寫入 customEvents

## 6️⃣ 突破 256KB 限制 — Client 端完整日誌

APIM 的 `largeLanguageModel.responses.maxSizeInBytes` 上限是 **256KB**（Azure 硬限制，無法調高）。Reasoning model 的 streaming SSE 容易超過這個大小，導致最終 `content` 被截斷。

**解法：** 在 client 端把完整 streaming 結果直接送到 Application Insights 的 `customEvents`。每個 property 上限 8KB 但可分多個 property，且總大小遠大於 256KB。

In [ ]:
# !pip install opencensus-ext-azure
import logging
from opencensus.ext.azure.log_exporter import AzureEventHandler

AI_CONNECTION_STRING = (
    "InstrumentationKey=a79e32b5-0613-4b78-ad01-c973a97210eb;"
    "IngestionEndpoint=https://eastus2-3.in.applicationinsights.azure.com/"
)

logger = logging.getLogger('ai-gw-client-logger')
logger.setLevel(logging.INFO)
if not any(isinstance(h, AzureEventHandler) for h in logger.handlers):
    logger.addHandler(AzureEventHandler(connection_string=AI_CONNECTION_STRING))

def log_full_response(correlation_id, deployment, full_content, full_reasoning, usage, finish_reason, is_streaming):
    """把完整 LLM 回應寫到 App Insights customEvents（不受 APIM 256KB 限制）"""
    def chunk(s, size=8000):
        return [s[i:i+size] for i in range(0, len(s), size)] if s else []
    props = {
        'correlationId': correlation_id,
        'deployment': deployment,
        'isStreaming': str(is_streaming),
        'finishReason': finish_reason or '',
        'usage': str(usage) if usage else '',
        'contentLength': str(len(full_content or '')),
        'reasoningLength': str(len(full_reasoning or '')),
    }
    for i, c in enumerate(chunk(full_content or '')):
        props[f'content_{i:02d}'] = c
    for i, r in enumerate(chunk(full_reasoning or '')):
        props[f'reasoning_{i:02d}'] = r
    logger.info('FullLLMResponse', extra={'custom_dimensions': props})
    print(f'\u2713 logged to App Insights (correlationId={correlation_id}, content={len(full_content or "")}B, reasoning={len(full_reasoning or "")}B)')


In [ ]:
# TC-06: Streaming + client-side full logging
import uuid
corr_id = f'streamtest-{uuid.uuid4().hex[:8]}'
print(f'correlationId = {corr_id}')

stream = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[
        {'role': 'user', 'content': '請用繁體中文簡單回應：今天天氣很好'},
    ],
    max_tokens=4000,
    stream=True,
    stream_options={'include_usage': True},
    extra_headers={'x-correlation-id': corr_id},
)

full_content = ''
full_reasoning = ''
usage_obj = None
finish_reason = None
for chunk in stream:
    if chunk.choices:
        delta = chunk.choices[0].delta
        if getattr(delta, 'content', None):
            full_content += delta.content
        r = getattr(delta, 'reasoning_content', None) or delta.model_dump().get('reasoning_content')
        if r:
            full_reasoning += r
        if chunk.choices[0].finish_reason:
            finish_reason = chunk.choices[0].finish_reason
    if chunk.usage:
        usage_obj = chunk.usage.model_dump()

print('--- Final Content ---')
print(full_content)
print(f'\n--- Reasoning length: {len(full_reasoning)} ---')
print(f'Usage: {usage_obj}')

# 重要：把完整結果送到 App Insights，繞過 APIM 的 256KB 限制
log_full_response(corr_id, DEPLOYMENT_NAME, full_content, full_reasoning, usage_obj, finish_reason, True)


### 查詢 Client 端完整日誌

等 1-2 分鐘後到 Log Analytics：

```kusto
AppEvents
| where TimeGenerated > ago(15m)
| where Name == 'FullLLMResponse'
| extend P = parse_json(tostring(Properties))
| project TimeGenerated,
          CorrelationId = tostring(P.correlationId),
          Deployment    = tostring(P.deployment),
          ContentLength = toint(P.contentLength),
          ReasoningLen  = toint(P.reasoningLength),
          // 把分塊欄位拼回來
          FullContent   = strcat(tostring(P.content_00), tostring(P.content_01), tostring(P.content_02)),
          FullReasoning = strcat(tostring(P.reasoning_00), tostring(P.reasoning_01), tostring(P.reasoning_02), tostring(P.reasoning_03)),
          Usage         = tostring(P.usage)
| order by TimeGenerated desc
```